In [ ]:
import pandas as pd
import numpy as np
import json
import logging
from typing import List, Tuple, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import warnings
import torch
warnings.filterwarnings('ignore')

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [ ]:
file_path = '../data/efsa_sentiment_classification.csv'
df = pd.read_csv(file_path)

df = df[:10]

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

SENTIMENT_POLARITIES = ['POS', 'NEG', 'NEU']
BATCH_SIZE = 10

FINANCIAL_EVENT_LIST = {
    "Financial Affairs": ["Profit Announcement", "Profit Forecast", "Other Financial Affairs"],
    "Shareholder Affairs": ["Stock Holding Adjustment", "Shareholder Pledge", "Release of Pledge", "Other Shareholder Affairs"],
    "Stock Affairs": ["Stock Price Movement", "Equity Incentives & Employee Stock Ownership Plans", "Stock Dividend", "Stock Buyback", "Stock Status", "Restricted Shares Release", "Other Stock Affairs"],
    "Business Operations": ["Product Dynamics", "Capacity Changes", "Initiating Cooperation", "Technical Quality Control, Qualification Changes", "Government Subsidies", "New Company Establishment", "Institutional Research", "Intellectual Property", "Sales, Market Share Changes", "Project Bidding", "Project Dynamics", "Other Business Operations Affairs"],
    "Compliance and Credit": ["Company Litigation", "Rating Adjustment", "Legal Affairs", "Clarification Announcements", "Regulatory Inquiries", "Case Investigations", "Administrative Penalties", "Other Compliance and Credit Affairs"],
    "Management Affairs": ["Employee Dynamics", "Directors, Supervisors, and Senior Executives Dynamics"],
    "Financing and Investment": ["Company Listing", "Mergers and Acquisitions", "Investment Events", "Stock Issuance", "Financing and Margin Trading", "Capital Flows", "Other Financing and Investment Affairs"]
}

FINE_TO_COARSE = {}
FINE_EVENT_LIST = []

for coarse_key, fine_list in FINANCIAL_EVENT_LIST.items():
    for fine_event in fine_list:
        FINE_TO_COARSE[fine_event] = coarse_key
        FINE_EVENT_LIST.append(fine_event)

INDUSTRY_LIST = df['Industry'].unique().tolist()

coarse_event_definitions = """
- Financial Affairs: Earnings announcements, forecasts, or other financial updates.
- Shareholder Affairs: Actions by shareholders such as pledges, stock adjustments.
- Stock Affairs: Stock buybacks, dividends, price movement announcements.
- Business Operations: New products, partnerships, quality control, or business activities.
- Compliance and Credit: Legal issues, investigations, rating changes.
- Management Affairs: Executive hires, promotions, or organizational changes.
- Financing and Investment: Mergers, acquisitions, funding rounds, IPOs.
"""

In [ ]:
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", trust_remote_code=True).half()
print("Model loaded successfully!")

Loading model and tokenizer...


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

ImportError: Using `low_cpu_mem_usage=True` or a `device_map` requires Accelerate: `pip install accelerate`

In [ ]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
logger = logging.getLogger(__name__)


def query_model(model, tokenizer, prompt: str, max_new_tokens=512, temperature=0.7) -> str:
    """Generate model response for given prompt using Mistral Instruct format."""
    instruction = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(instruction, return_tensors="pt").to(model.device)
    try:
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                    temperature=temperature, pad_token_id=tokenizer.eos_token_id)
        response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        # Remove prompt from response to get just the answer:
        answer = response.split(prompt)[-1].strip()
        return answer
    except Exception as e:
        logger.error(f"Error querying model: {e}")
        return ""


def extract_companies(text: str) -> str:
    """Extract company names from financial text."""
    prompt = (f"Financial text: {text}\n\n"
              "As a financial analyst, identify ALL company names mentioned in this text. "
              "If multiple companies are mentioned, separate them with commas. "
              "If only one company is mentioned, just provide that company name. "
              "Only answer with unique company names in full, not names of individuals, without additional text or punctuation."
              )

    try:
        response = query_model(model, tokenizer, prompt, temperature=0.2)
        companies = [company.strip()
                     for company in response.split(',') if company.strip()]
        return companies if companies else ["Unknown Company"]
    except Exception as e:
        logger.error(f"Error extracting company name: {e}")
        return "Unknown Company"


def get_industry(text: str, company_name: str) -> str:
    """Get industry for a company using model prompt."""
    prompt = (f"You are a financial analyst using professional databases to classify companies. "
              f"Text context: {text}\n"
              f"Company: {company_name}\n\n"
              f"As if consulting Bloomberg Terminal, FactSet, or Yahoo Finance, determine {company_name}'s industry.\n\n"
              # f"Analysis framework:\n"
              # f"1. Primary Business: What does {company_name} mainly do for revenue?\n"
              # f"2. Database Classification: How would major financial platforms categorize this company?\n"
              f"3. GICS Mapping: Which GICS sector best fits?\n\n"
              f"Available GICS sectors: {INDUSTRY_LIST}\n\n"
              # f"Key considerations:\n"
              # f"- Primary revenue source\n"
              # f"- Standard financial database classifications\n"
              # f"- How institutional investors would categorize this stock\n"
              # f"- SEC filing industry classifications\n\n"
              f"Based on professional financial database standards, select the most appropriate GICS sector.\n"
              "Respond only with the industry name from the list, without additional text or punctuation."
              )

    try:
        response = query_model(model, tokenizer, prompt, temperature=0.4)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Industry"
    except Exception as e:
        logger.error(f"Error getting industry for {company_name}: {e}")
        return "Unknown Industry"


def classify_coarse_grained_event(text: str, industry: str, company_name: str) -> Tuple[str, Any]:
    """Classify the coarse-grained event type."""
    coarse_events = list(FINANCIAL_EVENT_LIST.keys())
    prompt = (f"You are a financial sentiment model.\n\n"
              f"Text: \"{text}\"\n"
              f"Company: {company_name}\n"
              f"Industry: {industry}\n\n"
              f"Choose the most appropriate coarse-grained event type from this list based on what is explicitly stated in the text: {coarse_events}\n\n"
              f"The following are the event definitions: {coarse_event_definitions}"
              "Please only answer with one coarse-grained event type, without additional text or punctuation."
              )

    try:
        response = query_model(model, tokenizer, prompt, temperature=0.1)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Event"
    except Exception as e:
        logger.error(f"Error classifying coarse event: {e}")
        return "Unknown Event"


def classify_fine_grained_event(text: str, industry: str, company_name: str, coarse_event: str) -> Tuple[str, Any]:
    """Classify the fine-grained event type."""
    if coarse_event in FINANCIAL_EVENT_LIST:
        fine_events = FINANCIAL_EVENT_LIST[coarse_event]
        prompt = (f"You are a financial sentiment model. The following news text was classified as the event category: {coarse_event}.\n\n"
                  f"Text: \"{text}\"\n"
                  f"Company: {company_name}\n"
                  f"Industry: {industry}\n\n"
                  f"Now select the specific fine-grained event that best describes what happened, based on this list {fine_events}.\n\n"
                  "If the event doesn't fit perfectly into a specific event type, choose the most relevant 'Other' category.\n"
                  "Please only answer with the fine-grained event type, without additional text or punctuation."
                  )
    else:
        return "Unknown Fine Event"

    try:
        response = query_model(model, tokenizer, prompt, temperature=0.3)
        response = response.replace('"', '').replace("'", "")
        return response.strip() or "Unknown Fine Event"
    except Exception as e:
        return "Unknown Fine Event"


def classify_fine_grained_event_first(text: str, industry: str, company_name: str) -> Tuple[str, str, Any]:
    """Classify fine-grained event first, then map to coarse."""

    prompt = (f"Financial text: {text}\n"
              f"Company: {company_name}\n"
              f"Industry: {industry}\n\n"
              f"As a financial sentiment model, determine the specific type of corporate event that occurred at {company_name}. "
              f"Select the most precise event type from this comprehensive list: {FINE_EVENT_LIST}\n\n"
              "If none of the event types are explicity mentioned in the text, choose the most relevant 'Other' category.\n"
              "Respond only with an event type from the provided list, without additional text, quotation marks, or punctuation."
              )

    try:
        fine_response = query_model(
            model, tokenizer, prompt, temperature=0.5).strip()

        # Clean quotation marks
        fine_response = fine_response.replace('"', '').replace("'", "")

        # Map to coarse event
        coarse_event = FINE_TO_COARSE.get(fine_response, "Unknown Event")

        return coarse_event, fine_response
    except Exception as e:
        logger.error(f"Error classifying fine-grained event first: {e}")
        return "Unknown Event", "Unknown Fine Event"


def analyze_sentiment(text: str, industry: str, company_name: str, coarse_event: str, fine_event: str) -> str:
    """Analyze sentiment polarity of the event."""
    prompt = (f"You are a financial sentiment model. Analyze the sentiment expressed in the text.\n\n"
              f"Company: {company_name} ({industry})\n"
              f"Event: {fine_event}\n"
              f"News: {text}\n\n"
              f"- POS = The sentence includes positive or optimistic language.\n"
              f"- NEG = The sentence includes negative or pessimistic language.\n"
              "Please only answer with one of: POS, NEU, or NEG, without additional text or punctuation."
              )

    try:
        response = query_model(model, tokenizer, prompt, temperature=0.5)
        return response.strip() or "Unknown"
    except Exception as e:
        logger.error(f"Error analyzing sentiment: {e}")
        return "Unknown"

In [ ]:
def analyze_financial_text(text: str) -> Tuple[str, str, str, str, str]:
    """
    Analyze a single financial text and return the quintuple.

    Returns:
        Tuple[str, str, str, str, str]: (company_name, industry, coarse_event, fine_event, sentiment)
    """
    try:
        # Extract all company names
        company_names = extract_companies(text)
        results = []

        for company_name in company_names:
            try:
                # Get industry
                industry = get_industry(text, company_name)

                # Classify fine-grained event first, then derive coarse event
                coarse_event = classify_coarse_grained_event(
                    text, company_name, industry)

                fine_event = classify_fine_grained_event(
                    text, industry, company_name, coarse_event)

                # Analyze sentiment
                sentiment = analyze_sentiment(
                    text, industry, company_name, coarse_event, fine_event)

                results.append((text, company_name, industry,
                               coarse_event, fine_event, sentiment))

            except Exception as e:
                logger.error(
                    f"Error analyzing text for company {company_name}: {e}")
                results.append(
                    (company_name, "Error", "Error", "Error", "Error"))

        return results

    except Exception as e:
        logger.error(f"Error analyzing text: {e}")
        return [("Error", "Error", "Error", "Error", "Error")]

In [ ]:
results = []
# df = df[:100]
# Process each text
for idx, row in df.iterrows():
    text = row['Sentence']

    if pd.isna(text) or text.strip() == '':
        logger.warning(f"Empty text at index {idx}")
        results.append(("", "", "", "", "", ""))
        continue

    print(f"Processing {idx+1}/{len(df)}: {text[:50]}...")

    # Analyze the text
    result = analyze_financial_text(text)

    # Store results
    results.append(result)

    # Print progress every 10 items
    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1}/{len(df)} texts")

In [ ]:
# Flatten the list of lists and add the original sentence to each entry
flat_results_with_sentence = []
for i, sentence_results in enumerate(results):
    for company_result in sentence_results:
        flat_results_with_sentence.append(company_result)

results_df = pd.DataFrame(flat_results_with_sentence, columns=[
                          'Sentence', 'Company', 'Industry', 'Coarse Event', 'Fine Event', 'Sentiment'])
results_df['Sentiment'] = results_df['Sentiment'].str[:3]

2025-07-15 21:13:45,283 - ERROR - Error extracting company name: 'BertModel' object has no attribute 'chat'
2025-07-15 21:13:45,288 - ERROR - Error analyzing text: get_industry() missing 1 required positional argument: 'company_name'
2025-07-15 21:13:45,289 - ERROR - Error extracting company name: 'BertModel' object has no attribute 'chat'
2025-07-15 21:13:45,289 - ERROR - Error analyzing text: get_industry() missing 1 required positional argument: 'company_name'
2025-07-15 21:13:45,290 - ERROR - Error extracting company name: 'BertModel' object has no attribute 'chat'
2025-07-15 21:13:45,290 - ERROR - Error analyzing text: get_industry() missing 1 required positional argument: 'company_name'
2025-07-15 21:13:45,290 - ERROR - Error extracting company name: 'BertModel' object has no attribute 'chat'
2025-07-15 21:13:45,291 - ERROR - Error analyzing text: get_industry() missing 1 required positional argument: 'company_name'
2025-07-15 21:13:45,291 - ERROR - Error extracting company name:

Processing 1/10: Royal Mail chairman Donald Brydon set to step down...
Processing 2/10: Slump in Weir leads FTSE down from record high...
Processing 3/10: AstraZeneca wins FDA approval for key new lung can...
Processing 4/10: UPDATE 1-Lloyds to cut 945 jobs as part of 3-year ...
Processing 5/10: Standard Chartered Shifts Emerging-Markets Strateg...
Processing 6/10: AstraZeneca's Lung Cancer Drug Tagrisso Gets FDA A...
Processing 7/10: Severn Trent share price rises as first half profi...
Processing 8/10: Glencore sees Tripoli-based NOC as sole legal sell...
Processing 9/10: Lloyds to cut 945 jobs as part of three-year restr...
Processing 10/10: AstraZeneca to Buy ZS Pharma for $2.7 Billion...
Completed 10/10 texts


In [ ]:
results_df.head(20)

In [ ]:
df[:30]

In [ ]:
def calculate_accuracy_with_mismatch_lists(df, results_df):
    """
    Compare ground truth df with prediction results_df and calculate accuracy.

    Returns:
    tuple: (accuracy, mismatch_lists, matched_count, total_count)
    """
    # Initialize tracking variables
    matched_count = 0
    total_count = len(df)

    # Initialize mismatch lists
    mismatch_lists = {
        'company_mismatch': [],
        'sentiment_mismatch': [],
        'industry_mismatch': [],
        'coarse_event_mismatch': [],
        'fine_event_mismatch': [],
        'no_prediction': []
    }

    # Group original df by sentence to handle multiple rows per sentence
    grouped_original = df.groupby('Sentence')

    # Iterate through each sentence in the original dataframe
    for sentence, original_group in grouped_original:
        # Get all prediction rows for this sentence
        prediction_rows = results_df[results_df['Sentence'] == sentence]

        # Check if there are any predictions for this sentence
        if prediction_rows.empty:
            # Add all rows from this sentence to no_prediction list
            for _, original_row in original_group.iterrows():
                mismatch_info = {
                    'Sentence': original_row['Sentence'],
                    'Ground_Truth_Company': original_row['Company'],
                    'Ground_Truth_Sentiment': original_row['Sentiment'],
                    'Ground_Truth_Industry': original_row['Industry'],
                    'Ground_Truth_Coarse_Event': original_row['Coarse-Grained Event'],
                    'Ground_Truth_Fine_Event': original_row['Fine-Grained Event'],
                    'Predicted_Company': None,
                    'Predicted_Sentiment': None,
                    'Predicted_Industry': None,
                    'Predicted_Coarse_Event': None,
                    'Predicted_Fine_Event': None
                }
                mismatch_lists['no_prediction'].append(mismatch_info)
            continue

        # For each row in the original group (ground truth)
        for _, original_row in original_group.iterrows():
            found_match = False

            # Check if there's a matching company in predictions
            company_matches = prediction_rows[prediction_rows['Company'].str.lower(
            ).str.contains(original_row['Company'].lower(), na=False)]

            if not company_matches.empty:
                for _, pred_row in company_matches.iterrows():
                    sentiment_match = original_row['Sentiment'] == pred_row['Sentiment']
                    industry_match = pred_row['Industry'].lower(
                    ) in original_row['Industry'].lower()
                    coarse_match = original_row['Coarse-Grained Event'] == pred_row['Coarse Event']
                    fine_match = original_row['Fine-Grained Event'] == pred_row['Fine Event']

                    # Full quintuple match
                    if sentiment_match and industry_match and coarse_match and fine_match:
                        found_match = True
                        matched_count += 1
                        break

                # If company matches but other elements don't, record specific mismatches
                if not found_match:
                    # Take first company match
                    best_company_match = company_matches.iloc[0]

                    mismatch_info = {
                        'Sentence': original_row['Sentence'],
                        'Ground_Truth_Company': original_row['Company'],
                        'Ground_Truth_Sentiment': original_row['Sentiment'],
                        'Ground_Truth_Industry': original_row['Industry'],
                        'Ground_Truth_Coarse_Event': original_row['Coarse-Grained Event'],
                        'Ground_Truth_Fine_Event': original_row['Fine-Grained Event'],
                        'Predicted_Company': best_company_match['Company'],
                        'Predicted_Sentiment': best_company_match['Sentiment'],
                        'Predicted_Industry': best_company_match['Industry'],
                        'Predicted_Coarse_Event': best_company_match['Coarse Event'],
                        'Predicted_Fine_Event': best_company_match['Fine Event']
                    }

                    # Check each element and add to appropriate mismatch list
                    if original_row['Sentiment'] != best_company_match['Sentiment']:
                        mismatch_lists['sentiment_mismatch'].append(
                            mismatch_info)
                    if original_row['Industry'] != best_company_match['Industry']:
                        mismatch_lists['industry_mismatch'].append(
                            mismatch_info)
                    if original_row['Coarse-Grained Event'] != best_company_match['Coarse Event']:
                        mismatch_lists['coarse_event_mismatch'].append(
                            mismatch_info)
                    if original_row['Fine-Grained Event'] != best_company_match['Fine Event']:
                        mismatch_lists['fine_event_mismatch'].append(
                            mismatch_info)

            else:
                # Company doesn't match
                best_prediction = prediction_rows.iloc[0] if not prediction_rows.empty else None

                mismatch_info = {
                    'Sentence': original_row['Sentence'],
                    'Ground_Truth_Company': original_row['Company'],
                    'Ground_Truth_Sentiment': original_row['Sentiment'],
                    'Ground_Truth_Industry': original_row['Industry'],
                    'Ground_Truth_Coarse_Event': original_row['Coarse-Grained Event'],
                    'Ground_Truth_Fine_Event': original_row['Fine-Grained Event'],
                    'Predicted_Company': best_prediction['Company'] if best_prediction is not None else None,
                    'Predicted_Sentiment': best_prediction['Sentiment'] if best_prediction is not None else None,
                    'Predicted_Industry': best_prediction['Industry'] if best_prediction is not None else None,
                    'Predicted_Coarse_Event': best_prediction['Coarse Event'] if best_prediction is not None else None,
                    'Predicted_Fine_Event': best_prediction['Fine Event'] if best_prediction is not None else None
                }

                mismatch_lists['company_mismatch'].append(mismatch_info)

    # Calculate accuracy
    accuracy = matched_count / total_count

    return accuracy, mismatch_lists, matched_count, total_count


def create_mismatch_dataframes(mismatch_lists):
    """
    Convert mismatch lists to DataFrames for easier analysis.
    """
    mismatch_dfs = {}

    for mismatch_type, mismatch_list in mismatch_lists.items():
        mismatch_dfs[mismatch_type] = pd.DataFrame(mismatch_list)

    return mismatch_dfs


def analyze_mismatches_by_col(df, results_df):

    # Get quintuple accuracy with mismatch lists
    accuracy, mismatch_lists, matched_count, total_count = calculate_accuracy_with_mismatch_lists(
        df, results_df)

    print("=== QUINTUPLE ACCURACY ===")
    print(f"Total ground truth rows: {total_count}")
    print(f"Full quintuple matches: {matched_count}")
    print(f"Overall quintuple accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print()

    # Convert to DataFrames
    mismatch_dfs = create_mismatch_dataframes(mismatch_lists)

    print("=== MISMATCH PCT ===")
    for mismatch_type, mismatch_df in mismatch_dfs.items():
        count = len(mismatch_df)
        percentage = (count / total_count * 100) if total_count > 0 else 0

        print(f"{mismatch_type}: {count} ({percentage:.2f}%)")
        print(f"\t Accuracy: {((total_count - count) / total_count * 100)}")

    print()

    return accuracy, mismatch_dfs, matched_count, total_count

In [ ]:
accuracy, mismatch_dfs, matched_count, total_count = analyze_mismatches_by_col(
    df, results_df)

In [ ]:
mismatch_dfs['company_mismatch']